In [1]:
!pip install torch_geometric
!pip install adamp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for adamp: filename=adamp-0.3.0-py3-none-any.whl size=5984 sha256=c8983c0863ddf39df77de2e31400fa7c4323f4fc9a7c4f0d386b8453d920f78e
  Stored in directory: /root/.cache/pip/wheels/c7/ad/0f/b41b1c45b18c66e5eef5d2254415af8055c7e2b0934145157d
Successfully built adamp


In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool, global_max_pool, BatchNorm
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from torch.nn import Dropout, Linear
import numpy as np
import random
from torch.optim.lr_scheduler import CosineAnnealingLR

# Set random seeds for reproducibility
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# GIN model with batch normalization and dropout
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=5, dropout=0.5):
        super(GIN, self).__init__()

        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # Initial layer
        self.layers.append(
            GINConv(
                torch.nn.Sequential(
                    Linear(in_channels, hidden_channels),
                    torch.nn.ReLU(),
                    Linear(hidden_channels, hidden_channels)
                )
            )
        )
        self.norms.append(BatchNorm(hidden_channels))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(
                GINConv(
                    torch.nn.Sequential(
                        Linear(hidden_channels, hidden_channels),
                        torch.nn.ReLU(),
                        Linear(hidden_channels, hidden_channels)
                    )
                )
            )
            self.norms.append(BatchNorm(hidden_channels))

        self.dropout = Dropout(dropout)
        self.out = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = data.batch

        # Apply GIN layers with batch normalization
        for conv, norm in zip(self.layers, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = self.dropout(x)

        # Global mean pooling
        x = global_mean_pool(x, batch)
        return F.log_softmax(self.out(x), dim=1)

# Train the model with improved optimization and early stopping
def train_model(model, loader, epochs=200, lr=0.001, weight_decay=1e-4, patience=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    early_stopping_counter = 0
    best_loss = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        scheduler.step()

        # Early stopping mechanism
        if avg_loss < best_loss:
            best_loss = avg_loss
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1

        if early_stopping_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    return model

# Evaluation function
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    current_num_classes = out_prob.shape[1]
    if current_num_classes != num_classes:
        if current_num_classes < num_classes:
            padding = num_classes - current_num_classes
            out_prob = torch.cat([out_prob, torch.zeros(out_prob.size(0), padding)], dim=1)
        else:
            out_prob = out_prob[:, :num_classes]

    auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted') if num_classes == 2 else roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')
    logloss = log_loss(all_labels.cpu().numpy(), out_prob)

    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# Set random seed
set_random_seed(42)

# LOAD AND AUGMENT DATASETS
dataset_proteins = TUDataset(root='/content/PROTEINS', name='PROTEINS', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# SPLIT DATASETS INTO TRAINING AND TESTING SETS (70/30)
train_idx_proteins, test_idx_proteins = train_test_split(range(len(dataset_proteins)), test_size=0.3, random_state=42)
train_dataset_proteins = dataset_proteins[train_idx_proteins]
test_dataset_proteins = dataset_proteins[test_idx_proteins]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_proteins = DataLoader(train_dataset_proteins, batch_size=32, shuffle=True)
test_loader_proteins = DataLoader(test_dataset_proteins, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# INITIALIZE AND TRAIN MODEL ON PROTEINS
model_proteins = GIN(in_channels=dataset_proteins.num_node_features, hidden_channels=128, out_channels=dataset_proteins.num_classes, dropout=0.5)
train_model(model_proteins, train_loader_proteins, epochs=200)

# EVALUATE ON PROTEINS DATASET
print("\nResults on PROTEINS Dataset:")
proteins_results = evaluate_model(model_proteins, test_loader_proteins, num_classes=dataset_proteins.num_classes)
print(proteins_results)

# TRANSFER LEARNING TO MUTAG DATASET
model_mutag = GIN(in_channels=dataset_mutag.num_node_features, hidden_channels=128, out_channels=dataset_mutag.num_classes, dropout=0.5)
state_dict_proteins = model_proteins.state_dict()
state_dict_mutag = model_mutag.state_dict()

for key in state_dict_proteins:
    if key in state_dict_mutag and state_dict_proteins[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_proteins[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)
train_model(model_mutag, train_loader_mutag, epochs=200)

# EVALUATE ON MUTAG DATASET
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)




Results on PROTEINS Dataset:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


{'accuracy': 0.6826347305389222, 'f1': 0.6831176173272535, 'precision': 0.68364938811697, 'recall': 0.6826347305389222, 'auc_roc': 0.6998626373626373, 'log_loss': 0.7296040868370733}
Early stopping at epoch 89

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8771929824561403, 'f1': 0.8709881972712582, 'precision': 0.8824070666175929, 'recall': 0.8771929824561403, 'auc_roc': 0.9715099715099714, 'log_loss': 0.2354779966385444}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [ ]:
87,87,